# RAG + Contradiction Detection Pipeline
**Local version** — adapted from Google Colab original.

### Folder structure expected:
```
project/
├── dataset/
│   ├── prs/        ← put your PDF files here
│   └── text/       ← extracted .txt files will appear here (auto-created)
└── Gen_Datasets_Local.ipynb
```
Edit `INPUT_DIR` and `OUTPUT_DIR` in Cell 2 if your folders are elsewhere.

## Step 1 — Install dependencies

In [1]:
# Run once to install required packages
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu transformers torch pdfplumber nltk

Defaulting to user installation because normal site-packages is not writeable


## Step 2 — Import libraries & download NLTK data

In [2]:
import os

# Keep Hugging Face on the PyTorch backend. TensorFlow is installed in this
# environment with an older protobuf runtime, which can break pipeline import.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import csv
import json
import re
from datetime import datetime
from pathlib import Path

import pdfplumber
import numpy as np
import faiss
import nltk

from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from transformers import pipeline

nltk.download("punkt")
nltk.download("punkt_tab")


C:\Users\aryan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Step 3 — Extract text from PDFs
Reads every `.pdf` from `INPUT_DIR`, extracts text page by page, and saves a `.txt` file to `OUTPUT_DIR`.

In [3]:
# ── Edit these two paths to match your local setup ──────────────────────────
INPUT_DIR  = "dataset/prs"   # folder containing your PDF files
OUTPUT_DIR = "dataset/text"  # extracted text files will be saved here
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

extracted = 0
for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".pdf"):
            path = os.path.join(root, file)
            text = ""

            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"

            out_file = os.path.join(OUTPUT_DIR, file.replace(".pdf", ".txt"))
            with open(out_file, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"Extracted: {file}")
            extracted += 1

print(f"\nDone. {extracted} PDF(s) processed.")

Extracted: Act10of2026UP.pdf
Extracted: Act11of2026HR.pdf
Extracted: Act16of2026KA.pdf
Extracted: Act18of2026KA.pdf
Extracted: Act1of2026CG.pdf
Extracted: Act1of2026JH.pdf
Extracted: Act1of2026PD.pdf
Extracted: Act20of2025AP.pdf
Extracted: Act21of2025GA.pdf
Extracted: Act22of2025AP.pdf
Extracted: Act24of2025HR.pdf
Extracted: Act27of2025CG.pdf
Extracted: Act2of2026AS.pdf
Extracted: Act2of2026GA.pdf
Extracted: Act2of2026OD.pdf
Extracted: Act32of2025AS.pdf
Extracted: Act32of2025KA.pdf
Extracted: Act3of2025AP.pdf
Extracted: Act3of2026AS.pdf
Extracted: Act3of2026KA.pdf
Extracted: Act3of2026KL.pdf
Extracted: Act3of2026UP.pdf
Extracted: Act4of2025AR.pdf
Extracted: Act4of2026HR.pdf
Extracted: Act4of2026UP.pdf
Extracted: Act53of2025AS.pdf
Extracted: Act54of2025AS.pdf
Extracted: Act5of2026UP.pdf
Extracted: Act6of2021AR.pdf
Extracted: Act6of2026AS.pdf
Extracted: Act9of2025AP.pdf
Extracted: Bills_of_Lading_Act_2025.pdf
Extracted: Carriage_Of_Goods_By_Sea_Act_2025.pdf
Extracted: Central_Excise_(A)_

## Step 4 — Chunk text into sentences

In [4]:
TEXT_DIR = OUTPUT_DIR

MIN_CHARS = 80
MAX_CHARS = 1400

def infer_doc_type(filename):
    name = filename.lower()
    if "bill" in name:
        return "bill"
    if "act" in name:
        return "act"
    if "mpr" in name or "policy" in name or "apr" in name:
        return "policy_report"
    return "policy_document"

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_long_sentence(sentence, max_chars=MAX_CHARS):
    sentence = clean_text(sentence)

    if len(sentence) <= max_chars:
        return [sentence]

    parts = re.split(r"(?<=[.;:])\s+", sentence)
    chunks_out = []
    current = ""

    for part in parts:
        candidate = f"{current} {part}".strip()

        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks_out.append(current)
            current = part[:max_chars]

    if current:
        chunks_out.append(current)

    return chunks_out

chunk_records = []

for file in sorted(os.listdir(TEXT_DIR)):
    if file.endswith(".txt"):
        path = os.path.join(TEXT_DIR, file)
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
        sentences = sent_tokenize(text)
        local_chunk_id = 0

        for sentence in sentences:
            for chunk_text in split_long_sentence(sentence):
                if len(chunk_text) >= MIN_CHARS:
                    chunk_records.append({
                        "chunk_id": len(chunk_records),
                        "local_chunk_id": local_chunk_id,
                        "text": chunk_text,
                        "source_file": file,
                        "source_path": str(Path(path).resolve()),
                        "doc_id": Path(file).stem,
                        "doc_type": infer_doc_type(file),
                        "char_count": len(chunk_text),
                    })
                    local_chunk_id += 1

chunks = [record["text"] for record in chunk_records]

print(f"Total document chunks: {len(chunks)}")
print(f"Total source documents: {len({record['source_file'] for record in chunk_records})}")

if chunk_records:
    print("\nSample chunks with citations:")
    for record in chunk_records[:3]:
        print(f" - [{record['source_file']} #{record['local_chunk_id']}] {record['text'][:140]}")


Total document chunks: 38873
Total source documents: 168

Sample chunks with citations:
 - [APR_2024-25.txt #0] ANNUAL POLICY REVIEW April 2024 - March 2025 Annual Policy Review: April 2024 – March 2025 PRS Legislative Research PRS Legislative Research
 - [APR_2024-25.txt #1] You may choose to reproduce or redistribute this report for non-commercial purposes in part or in full to any other person with due acknowle
 - [APR_2024-25.txt #2] PRS makes every effort to use reliable and comprehensive information, but PRS does not represent that the contents of the report are accurat


## Step 5 — Load embedding model & encode chunks

In [5]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

if not chunks:
    raise ValueError(
        "No chunks found! Make sure your PDFs are in INPUT_DIR and "
        "text extraction ran successfully in Step 3."
    )

embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(f"\nEmbeddings shape: {embeddings.shape}")


C:\Users\aryan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aryan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 


Embeddings shape: (38873, 384)


## Step 6 — Build FAISS index

In [6]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(np.asarray(embeddings, dtype="float32"))

print(f"FAISS cosine index created with {index.ntotal} vectors.")


FAISS cosine index created with 38873 vectors.


## Step 7 — Define retrieval function

In [7]:
def retrieve(query, k=5, exclude_doc_id=None):
    """Return top-k semantically similar chunks with source metadata."""
    search_k = min(max(k * 5, k), len(chunks))

    q_embed = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = index.search(q_embed, search_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        record = dict(chunk_records[int(idx)])

        if exclude_doc_id and record["doc_id"] == exclude_doc_id:
            continue

        record["retrieval_score"] = float(score)
        results.append(record)

        if len(results) == k:
            break

    return results

def print_retrieval_results(results):
    for rank, record in enumerate(results, start=1):
        print(
            f"\nRank {rank} | score={record['retrieval_score']:.3f} | "
            f"{record['source_file']} #{record['local_chunk_id']}"
        )
        print(record["text"][:500])


## Step 8 — Load NLI model for contradiction detection

In [8]:
nli = pipeline(
    "text-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v1",
    truncation=True,
)

def normalise_nli_output(result):
    item = result[0] if isinstance(result, list) else result

    return {
        "label": item["label"].lower(),
        "score": float(item["score"]),
    }

def detect_contradiction(statement_a, statement_b):
    """Run NLI for a pair of policy clauses and return label + confidence."""
    result = nli(f"{statement_a} </s> {statement_b}")
    return normalise_nli_output(result)

def classify_conflict_type(a, b):
    """Simple demo classifier: direct / scope / temporal / indirect."""
    text = f"{a} {b}".lower()

    has_negative = re.search(
        r"\bshall not\b|\bmust not\b|\bprohibited\b|\bban(ned)?\b",
        text,
    )
    has_positive = re.search(
        r"\bshall\b|\bmust\b|\bmay\b|\ballowed\b|\bpermitted\b",
        text,
    )

    if has_negative and has_positive:
        return "direct"

    if re.search(r"\bexcept\b|\bunless\b|\bonly\b|\bsubject to\b|\bprovided that\b", text):
        return "scope"

    if re.search(r"\bbefore\b|\bafter\b|\bwithin\b|\bfrom\b|\buntil\b|\bdate\b", text):
        return "temporal"

    return "indirect"

def rag_contradiction(query, k=8, min_contradiction_score=0.50, exclude_doc_id=None):
    """Retrieve candidate clauses and keep likely contradictions with citations."""
    candidates = retrieve(query, k=k, exclude_doc_id=exclude_doc_id)
    results = []

    for candidate in candidates:
        nli_result = detect_contradiction(query, candidate["text"])

        is_contradiction = (
            "contrad" in nli_result["label"]
            and nli_result["score"] >= min_contradiction_score
        )

        results.append({
            "query": query,
            "candidate_text": candidate["text"],
            "source_file": candidate["source_file"],
            "source_path": candidate["source_path"],
            "doc_id": candidate["doc_id"],
            "doc_type": candidate["doc_type"],
            "chunk_id": candidate["chunk_id"],
            "local_chunk_id": candidate["local_chunk_id"],
            "retrieval_score": candidate["retrieval_score"],
            "nli_label": nli_result["label"],
            "nli_score": nli_result["score"],
            "is_contradiction": is_contradiction,
            "conflict_type": classify_conflict_type(query, candidate["text"])
            if is_contradiction
            else "none",
        })

    return results


C:\Users\aryan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aryan\.cache\huggingface\hub\models--MoritzLaurer--deberta-v3-base-zeroshot-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device 

## Step 9 — Run a query

In [9]:
query = "States shall provide free electricity subsidy to farmers"

output = rag_contradiction(query, k=8, min_contradiction_score=0.50)

for row in output:
    print("\nQuery      :", row["query"])
    print("Source     :", f"{row['source_file']} #{row['local_chunk_id']}")
    print("Retrieved  :", f"{row['retrieval_score']:.3f}")
    print("NLI        :", f"{row['nli_label']} ({row['nli_score']:.3f})")
    print("Conflict   :", row["conflict_type"])
    print("Candidate  :", row["candidate_text"][:700])



Query      : States shall provide free electricity subsidy to farmers
Source     : MPR_March2023.txt #153
Retrieved  : 0.608
NLI        : not_entailment (0.998)
Conflict   : none
Candidate  : Schemes providing farmers with The Committee noted that there is a severe financial assistance and free or subsidised shortage of monitoring stations in the Indian electricity for irrigation have contributed to this Himalayan Region.

Query      : States shall provide free electricity subsidy to farmers
Source     : MPR-February_2024.txt #417
Retrieved  : 0.592
NLI        : not_entailment (0.907)
Conflict   : none
Candidate  : 68, ‘Promotion of Climate Resilient Farming’, Standing 54 The Electricity (Rights of Consumers) Amendment Rules, 2024, Committee on Agriculture, Animal Husbandry, and Food Processing, Ministry of Power, February 22, 2024, Lok Sabha, February 7, 2024, https://powermin.gov.in/sites/default/files/webform/notices/Electricit https://sansad.in/getFile/lsscommittee/Agriculture,%20

## Step 10 — Save FAISS index locally

In [10]:
INDEX_PATH = "faiss_index.bin"
METADATA_PATH = "faiss_metadata.json"

faiss.write_index(index, INDEX_PATH)

Path(METADATA_PATH).write_text(
    json.dumps(chunk_records, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"FAISS index saved to: {os.path.abspath(INDEX_PATH)}")
print(f"Chunk metadata saved to: {os.path.abspath(METADATA_PATH)}")


FAISS index saved to: c:\Users\aryan\OneDrive\Desktop\project\faiss_index.bin
Chunk metadata saved to: c:\Users\aryan\OneDrive\Desktop\project\faiss_metadata.json


STEP 11 - REPORT GENERATOR


In [11]:
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

def build_conflict_report(query, k=10, min_contradiction_score=0.50):
    rows = rag_contradiction(
        query,
        k=k,
        min_contradiction_score=min_contradiction_score,
    )

    contradictions = [row for row in rows if row["is_contradiction"]]

    report = {
        "query": query,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "retrieved_candidates": len(rows),
        "contradictions_found": len(contradictions),
        "thresholds": {
            "min_contradiction_score": min_contradiction_score,
            "top_k": k,
        },
        "results": contradictions,
        "all_candidates": rows,
    }

    return report

def save_conflict_report(report, basename="policy_conflict_report"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    json_path = REPORT_DIR / f"{basename}_{timestamp}.json"
    csv_path = REPORT_DIR / f"{basename}_{timestamp}.csv"

    json_path.write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    fieldnames = [
        "query",
        "source_file",
        "local_chunk_id",
        "doc_type",
        "retrieval_score",
        "nli_label",
        "nli_score",
        "conflict_type",
        "candidate_text",
        "source_path",
    ]

    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in report["results"]:
            writer.writerow({key: row.get(key, "") for key in fieldnames})

    return json_path, csv_path

report = build_conflict_report(query, k=10, min_contradiction_score=0.50)
json_path, csv_path = save_conflict_report(report)

print(f"Contradictions found: {report['contradictions_found']}")
print(f"JSON report: {json_path.resolve()}")
print(f"CSV report : {csv_path.resolve()}")


Contradictions found: 0
JSON report: C:\Users\aryan\OneDrive\Desktop\project\reports\policy_conflict_report_20260412_160647.json
CSV report : C:\Users\aryan\OneDrive\Desktop\project\reports\policy_conflict_report_20260412_160647.csv


STEP 12 - BATCH AUDIT

In [12]:
policy_claims = [
    "States shall provide free electricity subsidy to farmers",
    "All government examinations must be conducted only through offline mode",
]

batch_reports = [
    build_conflict_report(claim, k=10, min_contradiction_score=0.50)
    for claim in policy_claims
]

combined_report = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "claims_checked": len(policy_claims),
    "total_contradictions": sum(
        report["contradictions_found"] for report in batch_reports
    ),
    "reports": batch_reports,
}

json_path = REPORT_DIR / f"batch_policy_audit_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

json_path.write_text(
    json.dumps(combined_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Claims checked: {combined_report['claims_checked']}")
print(f"Total contradictions: {combined_report['total_contradictions']}")
print(f"Batch report: {json_path.resolve()}")


Claims checked: 2
Total contradictions: 0
Batch report: C:\Users\aryan\OneDrive\Desktop\project\reports\batch_policy_audit_20260412_160709.json
